<a href="https://colab.research.google.com/github/wzy34/colab_projects/blob/main/Titanic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import pandas as pd
from torch.utils.data import Dataset, DataLoader, TensorDataset, random_split
import numpy as np

In [ ]:
import kagglehub

kagglehub.login()

In [ ]:
import kagglehub

path = kagglehub.dataset_download("shuofxz/titanic-machine-learning-from-disaster")

print("Dataset path:", path)

100%|██████████| 33.1k/33.1k [00:00<00:00, 18.6MB/s]

Extracting files...
Dataset path: /root/.cache/kagglehub/datasets/shuofxz/titanic-machine-learning-from-disaster/versions/1


In [ ]:
import os

print(os.listdir(path))

['test.csv', 'train.csv']


In [ ]:
df = pd.read_csv(os.path.join(path, "train.csv"))
df.head(n = 40)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
5,6,0,3,"Moran, Mr. James",male,NaN,0,0,330877,8.4583,NaN,Q
6,7,0,1,"McCarthy, Mr. Timothy J",male,54.0,0,0,17463,51.8625,E46,S
7,8,0,3,"Palsson, Master. Gosta Leonard",male,2.0,3,1,349909,21.0750,NaN,S
8,9,1,3,"Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)",female,27.0,0,2,347742,11.1333,NaN,S
9,10,1,2,"Nasser, Mrs. Nicholas (Adele Achem)",female,14.0,1,0,237736,30.0708,NaN,C


In [ ]:
train_df = pd.read_csv(os.path.join(path, "train.csv"))
test_df = pd.read_csv(os.path.join(path, "test.csv"))

In [ ]:
train_df["Sex"] = train_df["Sex"].map({
    "male": 0,
    "female": 1
})

test_df["Sex"] = test_df["Sex"].map({
    "male": 0,
    "female": 1
})

In [ ]:
train_df = pd.get_dummies(
    train_df,
    columns=["Embarked"],
    dtype=float
)

test_df = pd.get_dummies(
    test_df,
    columns=["Embarked"],
    dtype=float
)

In [ ]:
train_df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked_C,Embarked_Q,Embarked_S
0,1,0,3,"Braund, Mr. Owen Harris",0,22.0,1,0,A/5 21171,7.2500,NaN,0.0,0.0,1.0
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",1,38.0,1,0,PC 17599,71.2833,C85,1.0,0.0,0.0
2,3,1,3,"Heikkinen, Miss. Laina",1,26.0,0,0,STON/O2. 3101282,7.9250,NaN,0.0,0.0,1.0
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",1,35.0,1,0,113803,53.1000,C123,0.0,0.0,1.0
4,5,0,3,"Allen, Mr. William Henry",0,35.0,0,0,373450,8.0500,NaN,0.0,0.0,1.0


In [ ]:
train_df["Age"] = train_df["Age"].fillna(train_df["Age"].median())
test_df["Age"] = test_df["Age"].fillna(train_df["Age"].median())

train_df["Fare"] = train_df["Fare"].fillna(train_df["Fare"].median())
test_df["Fare"] = test_df["Fare"].fillna(train_df["Fare"].median())

In [ ]:
features = [
    "Pclass",
    "Sex",
    "Age",
    "SibSp",
    "Parch",
    "Fare",
    "Embarked_C",
    "Embarked_Q",
    "Embarked_S"
]

X = train_df[features]
y = train_df["Survived"]
print(X.shape, y.shape)

(891, 9) (891,)


In [ ]:
X = torch.tensor(
    train_df[features].to_numpy(),
    dtype=torch.float32
)

y = torch.tensor(
    train_df["Survived"].to_numpy(),
    dtype=torch.float32
)
print(X.shape, y.shape)

dataset = TensorDataset(X, y)

torch.Size([891, 9]) torch.Size([891])


In [ ]:
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size

train_dataset, val_dataset = random_split(
    dataset,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

In [ ]:
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False
)

In [ ]:
from torch import nn

class TitanicModel(nn.Module):
  def __init__(self):
    super().__init__()
    self.linear = nn.Sequential(
        nn.Linear(9, 16),
        nn.ReLU(),
        nn.Linear(16, 8),
        nn.ReLU(),
        nn.Linear(8, 1)
    )

  def forward(self, x):
    return self.linear(x).squeeze(-1)


In [ ]:
model = TitanicModel()

In [ ]:
loss_fn = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [ ]:
def train(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    model.train()
    for batch, (X, y) in enumerate(dataloader):
        #X, y = X.to(device), y.to(device)

        # Compute prediction error
        pred = model(X)
        loss = loss_fn(pred, y)

        # Backpropagation
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if batch % 100 == 0:
            loss = loss.item()
            print(f"loss: {loss:>7f}")

In [ ]:
def test(dataloader, model, loss_fn):
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    model.eval()
    test_loss, correct = 0, 0
    with torch.no_grad():
        for X, y in dataloader:
            #X, y = X.to(device), y.to(device)
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += ((pred >= 0).float() == y).float().sum().item()
    test_loss /= num_batches
    correct /= size
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")

In [ ]:
epochs = 100
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train(train_loader, model, loss_fn, optimizer)
    test(val_loader, model, loss_fn)

Epoch 1
-------------------------------
loss: 0.921810
Test Error: 
 Accuracy: 61.5%, Avg loss: 0.690657 

Epoch 2
-------------------------------
loss: 0.669095
Test Error: 
 Accuracy: 70.9%, Avg loss: 0.644590 

Epoch 3
-------------------------------
loss: 0.640430
Test Error: 
 Accuracy: 68.2%, Avg loss: 0.632930 

Epoch 4
-------------------------------
loss: 0.610530
Test Error: 
 Accuracy: 67.6%, Avg loss: 0.618053 

Epoch 5
-------------------------------
loss: 0.590859
Test Error: 
 Accuracy: 68.7%, Avg loss: 0.614814 

Epoch 6
-------------------------------
loss: 0.617794
Test Error: 
 Accuracy: 69.3%, Avg loss: 0.608958 

Epoch 7
-------------------------------
loss: 0.622623
Test Error: 
 Accuracy: 68.7%, Avg loss: 0.606397 

Epoch 8
-------------------------------
loss: 0.699782
Test Error: 
 Accuracy: 68.7%, Avg loss: 0.602815 

Epoch 9
-------------------------------
loss: 0.564145
Test Error: 
 Accuracy: 69.8%, Avg loss: 0.596028 

Epoch 10
----------------------------

In [ ]:
test_loader = DataLoader(
    TensorDataset(
        torch.tensor(
            test_df[features].to_numpy(),
            dtype=torch.float32
        )
    ),
    batch_size=32,
    shuffle=False
)

In [ ]:
model.eval()

all_predictions = []

with torch.no_grad():
    for X_batch in test_loader:

        logits = model(X_batch[0])
        probabilities = torch.sigmoid(logits)

        predictions = (probabilities >= 0.5).int()

        all_predictions.append(predictions)

all_predictions = torch.cat(all_predictions)
print(all_predictions)

tensor([0, 0, 0, 0, 1, 0, 1, 0, 1, 0, 0, 0, 1, 0, 1, 1, 0, 0, 0, 1, 0, 0, 1, 1,
        1, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0,
        1, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 1, 1, 0, 0, 1, 1, 0,
        1, 1, 1, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 1, 0, 1, 0, 1, 0,
        1, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 0, 0, 1, 1, 1,
        1, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0,
        0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 1, 0, 1, 1, 0, 1, 0, 0, 1, 0, 0,
        1, 1, 0, 0, 0, 0, 0, 1, 1, 0, 1, 1, 0, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 1, 1, 0, 1, 1, 0, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0,
        1, 0, 1, 0, 1, 0, 1, 0, 1, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 1,
        1, 1, 0, 0, 0, 0, 1, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 1,
        0, 0, 0, 0, 1, 0, 1, 0, 1, 1, 0, 1, 0, 0, 0, 0, 1, 0, 1, 1, 1, 0, 0, 0,
        0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0,